In [41]:
import os
from pathlib import Path
from glob import glob
import pickle
import re

import pandas as pd
import numpy as np

In [42]:
from bikipy.behaviour.nort.experiment import ObjectRecognitionExperiment
from bikipy.plugins.belhaj import (
    get_animal_id_to_apparatus,
    get_exp_id_to_stage,
    get_exp_id_to_animal_id,
    get_animal_id_to_exp_ids,
    round_to_apparatus_to_general_object_fields,
)
from bikipy.utils.video import get_video_data

In [43]:
DEEPLABCUT_DIR = Path("/mnt/md0/Projects/Neuroscience/Imen/data/nort")
ROUND_DIR_NAME = "Round_1"

HABITUATION_DIR = DEEPLABCUT_DIR / "Open-Field" / ROUND_DIR_NAME
NOVELTY_DIR = DEEPLABCUT_DIR / "Novelty" / ROUND_DIR_NAME

ROOT_DIR = Path("/home/can/PycharmProjects/BiKiPy/examples/nort_belhaj_analysis")

EXP_ID_REGEX_PATTERN = re.compile("\d+")

In [44]:
# Set sheet name to 0 for Round_1 and 1 for Round_2
exp_metadata_df = pd.read_excel(str(ROOT_DIR / "nort_round_1.xlsx"), sheet_name=0, engine="openpyxl")

In [45]:
with open(ROOT_DIR / "area_images" / "A_annotations.pickle", "rb") as infile:
    round_to_field_apparatus = pickle.load(infile)
round_keys = [f"round_{num}" for num in range(len(round_to_field_apparatus))]
round_to_field_to_apparatus = {
    rem_round: round_to_apparatus_to_general_object_fields(field_apparatus)
    for rem_round, field_apparatus in zip(round_keys, round_to_field_apparatus.values())
}
field_to_apparatus = round_to_field_to_apparatus[f"round_{1 if '1' in ROUND_DIR_NAME else 2}"]

In [46]:
animal_id_to_app = get_animal_id_to_apparatus(exp_metadata_df, EXP_ID_REGEX_PATTERN)
exp_id_to_stage = get_exp_id_to_stage(exp_metadata_df, EXP_ID_REGEX_PATTERN)
exp_to_animal = get_exp_id_to_animal_id(get_animal_id_to_exp_ids(exp_metadata_df))

In [47]:
trial_id_range_to_exp_meta = {}
trial_id_to_coordinate_data_path = {}
for exp_class, root in zip(("habituation", "novelty"), (HABITUATION_DIR, NOVELTY_DIR)):
    for exp_dir in os.listdir(root):
        exp_designation = exp_dir.split("_")[0]
        exp_path = root / exp_dir
        glob_exp_data_path = exp_path / "**" if exp_class == "novelty" else exp_path

        trial_id_to_coordinate_data_path[exp_designation] = {}
        for data_path in glob(str(glob_exp_data_path / "*.h5")):
            exp_id = int(EXP_ID_REGEX_PATTERN.findall(Path(data_path).stem)[0])

            trial_id_to_coordinate_data_path[exp_designation][exp_id] = data_path

        trial_id_range_to_exp_meta[exp_designation] = {}
        for data_path in glob(str(glob_exp_data_path / "*.mp4")):
            exp_id = int(EXP_ID_REGEX_PATTERN.findall(Path(data_path).stem)[0])

            trial_id_range_to_exp_meta[exp_designation][exp_id] = {
                "stage": exp_class,
                "recording_resolution": (video_data := get_video_data(data_path))[1:3],
                "fps": video_data[3],
                "animal_id": (animal_id := exp_to_animal[exp_id]),
                "field": animal_id_to_app[animal_id],
            }

In [48]:
results = {
    exp_designation: ObjectRecognitionExperiment(
        trial_id_range_to_exp_meta=trial_id_range_to_exp_meta[exp_designation],
        metric_resolution=40,
        nose_label="nose",
        eye_center_label="mid-left_ear-right_ear",
        torso_label="mid-mid-left_ear-right_ear-tail",
        object_field_to_object_field_object=field_to_apparatus,
        center_metric_length=20,
        maximum_radians_inter_gaze_perimeter=1 / 4 * np.pi,
        trial_id_to_coordinate_data_path=trial_id_to_coordinate_data_path[exp_designation],
        midpoint_groups=[
            ("left_ear", "right_ear"),
            ("mid-left_ear-right_ear", "tail"),
        ],
    )
    for exp_designation in trial_id_range_to_exp_meta
}

In [ ]:
with pd.ExcelWriter(ROOT_DIR / "nort_analysis.ods", strings_to_formulas=False, strings_to_urls=False) as writer:
    for exp_designation, data in results.items():
        data.df.to_excel(writer, sheet_name=exp_designation)